# Upper-Stream Agent Workflow (LLM-Powered)

This notebook implements an autonomous agent that follows `prompts/upper_stream_agent.md` to:
1. Expand user queries
2. Harvest literature using `FullTextJournalSearchTool`
3. Extract chemical entities using `ChemicalNERTool`
4. Filter and prepare candidates for downstream MAS workflow

The agent is powered by LLM planning and reasoning, similar to `hmdb_test_agent.ipynb`.

In [1]:
import os
from typing import Optional

from dotenv import load_dotenv
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from pydantic import Field, SecretStr


os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

load_dotenv()

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                 openai_api_key: Optional[str] = None,
                 **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)

/home/chen/GNPS_SIM/agent/db_form_agent_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------- stdlib --------------------
from pathlib import Path
import re, logging, warnings, sqlite3, os, time, random, pickle, requests
import json
import copy
from typing import List, TypedDict, Optional, Dict

# -------------------- langchain / langgraph --------------------
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tracers.langchain import wait_for_all_tracers

# -------------------- local tools --------------------
import sys
sys.path.append('../code')
from chemical_ner_tool import ChemicalNerTool
from searching_tool import FullTextJournalSearchTool

# -------------------- logging --------------------
logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(name)s | %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
log = logging.getLogger("upper_stream_agent")

# -------------------- constants --------------------
MAX_ITERATIONS = 20

def _vprint(*args, **kwargs):
    if os.getenv("NODE_LOGS", "1") in ("1", "true", "TRUE", "yes", "YES"):
        print(*args, **kwargs)

## Load Upper-Stream Agent Prompt

In [3]:
PROMPTS_DIR = Path("/home/chen/GNPS_SIM/agent/db_form_agent_system/prompts")
UPPER_STREAM_PROMPT_PATH = PROMPTS_DIR / "upper_stream_agent_test.md"

UPPER_STREAM_AGENT_PROMPT = SystemMessage(content=UPPER_STREAM_PROMPT_PATH.read_text(encoding="utf-8"))

print("✓ Loaded upper-stream agent prompt")

✓ Loaded upper-stream agent prompt


## Initialize Models & Tools

In [4]:
def _init_models():
    agent_model = ChatOpenRouter(model_name="google/gemini-2.5-flash", temperature=0.0)
    return agent_model

def _build_tools(results_base_dir: str, min_date: str = None, max_date: str = None) -> List:
    """
    Build tools for the agent.
    
    Args:
        results_base_dir: Base directory for search results
        min_date: Minimum publication date (YYYY, YYYY/MM, or YYYY/MM/DD format)
        max_date: Maximum publication date (YYYY, YYYY/MM, or YYYY/MM/DD format)
    
    Returns:
        List of tools
    """
    ftjs = FullTextJournalSearchTool(
        results_base_dir=results_base_dir,
        include_pubmed=True,
        include_preprints=True,
        include_arxiv=False,
        include_medrxiv=True,
        include_eupmc=True,
        use_pubmed_free_fulltext_bias=True,
        max_parallel_downloads=6,
        min_date=min_date,  # Date range filter
        max_date=max_date,  # Date range filter
    )
    chem_ner = ChemicalNerTool()
    return [ftjs, chem_ner]

agent_model = _init_models()
tools = _build_tools("/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results")

print(f"✓ Initialized model: {agent_model.model_name}")
print(f"✓ Loaded {len(tools)} tools: {[t.name for t in tools]}")

✓ Initialized model: google/gemini-2.5-flash
✓ Loaded 2 tools: ['fulltext_journal_search', 'chemical_ner']


## Build Agent Executor

In [5]:
memory = MemorySaver()

agent_executor = create_react_agent(
    agent_model, 
    tools, 
    prompt=UPPER_STREAM_AGENT_PROMPT, 
    checkpointer=memory
)

print("✓ Agent executor ready")

✓ Agent executor ready


/tmp/ipykernel_1769714/3069743834.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


## State Definition

In [6]:
class AgentState(TypedDict):
    messages: List[HumanMessage | AIMessage]
    original_query: str
    search_session_dir: str
    iteration_count: int
    last_output: str | None
    ner_results_file: str | None  # Path to saved NER results

## Agent Node

In [7]:
def agent_node(state: AgentState, config: RunnableConfig):
    _vprint("\n----- Upper-Stream Agent Node Invoked -----")
    
    session_dir = state.get("search_session_dir")
    if not session_dir:
        raise ValueError("search_session_dir not found in state.")
    
    # Configure tool context
    tool_config = config.copy()
    tool_config["configurable"] = config.get("configurable", {}).copy()
    tool_config["configurable"]["search_session_dir"] = session_dir
    
    tags = list(tool_config.get("tags") or [])
    tags.append(f"SSD::{session_dir}")
    tags.append(f"search_session_dir={session_dir}")
    tool_config["tags"] = tags
    
    metadata = dict(tool_config.get("metadata") or {})
    metadata["search_session_dir"] = session_dir
    tool_config["metadata"] = metadata
    
    tool_config.pop("recursion_limit", None)
    
    # Seed ContextVar for tools
    try:
        FullTextJournalSearchTool.set_session_dir(session_dir)
    except Exception:
        pass
    
    input_messages = state["messages"]
    result = agent_executor.invoke({"messages": input_messages}, config=tool_config)
    
    ai_content = ""
    if 'messages' in result and isinstance(result['messages'], list) and result['messages']:
        ai_content = result['messages'][-1].content
    else:
        _vprint("[ERROR] Agent executor did not return expected messages list.")
        ai_content = '{"error": "Agent execution failed internally."}'
    
    new_ai_message = AIMessage(content=ai_content)
    all_messages = state["messages"] + [new_ai_message]
    
    return {
        "messages": all_messages,
        "iteration_count": state.get("iteration_count", 0) + 1,
        "last_output": ai_content,
    }

## Stopping Condition

In [8]:
def should_continue(state: AgentState):
    """Check if agent has completed the workflow or should continue."""
    
    last_output = state.get("last_output", "")
    iteration = state.get("iteration_count", 0)
    
    # Stop if max iterations reached
    if iteration >= MAX_ITERATIONS:
        _vprint(f"\n----- Stopping: Max iterations ({MAX_ITERATIONS}) reached -----")
        return END
    
    if last_output:
        json_obj = None
        if isinstance(last_output, str):
            try:
                json_obj = json.loads(last_output)
            except json.JSONDecodeError:
                json_obj = None
        elif isinstance(last_output, dict):
            json_obj = last_output
        
        if isinstance(json_obj, dict):
            has_results_file = any(key in json_obj for key in ("results_file", "ner_results_file"))
            has_candidates = "chemical_candidates" in json_obj or "candidates" in json_obj
            has_summary = "literature_summary" in json_obj or "session_directory" in json_obj
            has_queries = "expanded_queries" in json_obj
            mention_count = json_obj.get("mention_count")
            
            # Stop once the agent produces the NER deliverable
            if has_results_file and json_obj.get("error") not in (True, "true"):
                _vprint("\n----- Stopping: Agent produced NER deliverable -----")
                return END
            
            if mention_count and isinstance(mention_count, (int, float)):
                _vprint("\n----- Stopping: Agent reported mention count -----")
                return END
            
            if has_candidates and (has_summary or has_queries):
                _vprint("\n----- Stopping: Agent completed workflow with structured output -----")
                return END
        
        lower = last_output.lower() if isinstance(last_output, str) else ""
        
        # Text-based completion signals
        completion_phrases = [
            "workflow complete",
            "ready for mas",
            "deliverable",
            "handoff",
            "candidates prepared",
        ]
        
        if any(phrase in lower for phrase in completion_phrases):
            _vprint("\n----- Stopping: Agent signaled completion -----")
            return END
    
    # Continue if no stopping condition met
    return "agent"

## Build Workflow

In [9]:
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue)

app = workflow.compile(checkpointer=memory)

print("✓ Workflow compiled")

✓ Workflow compiled


## Helper Functions

In [10]:
def extract_final_output(messages):
    """Extract structured output from final agent message."""
    if not messages:
        return {}
    
    last_content = messages[-1].content if messages else ""
    
    try:
        return json.loads(last_content)
    except json.JSONDecodeError:
        # Try to extract JSON from text
        match = re.search(r'\{[\s\S]*\}', last_content)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    
    return {"raw_output": last_content}

## Run Agent

In [11]:
def run_upper_stream_agent(query: str, base_results_dir: str = None):
    """
    Run the upper-stream agent on a user query.
    
    Args:
        query: User request (e.g., "form a database of compounds found in hairs")
        base_results_dir: Base directory for results (default: searching_results/)
    
    Returns:
        dict: Final output with expanded_queries, literature_summary, chemical_candidates, etc.
    """
    if base_results_dir is None:
        base_results_dir = "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results"

    
    # Create session directory
    query_timestamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime())
    micros = f"{int(time.time_ns() % 1_000_000):06d}"
    nonce = f"{random.randint(0, 9999):04d}"

    os.environ["LANGCHAIN_PROJECT"] = f"Upper Agent Run - {query_timestamp}"
    
    session_dir_rel = f"upper_stream_{query_timestamp}_{micros}_{nonce}"
    session_dir_abs = f"{base_results_dir}/{session_dir_rel}"
    
    os.makedirs(session_dir_abs, exist_ok=True)
    _vprint(f"\n📁 Session directory: {session_dir_abs}")
    
    # Initialize state
    init_state: AgentState = {
        'messages': [HumanMessage(content=query)],
        'original_query': query,
        'search_session_dir': session_dir_abs,
        'iteration_count': 0,
        'last_output': None,
    }
    
    thread_id = f"upper_stream_{query_timestamp}_{random.randint(1000, 9999)}"
    cfg: RunnableConfig = {
        'configurable': {
            'thread_id': thread_id,
            'search_session_dir': session_dir_abs,
        },
        'recursion_limit': MAX_ITERATIONS,
    }
    
    # Seed tool context
    try:
        FullTextJournalSearchTool.set_session_dir(session_dir_abs)
    except Exception:
        pass
    
    _vprint(f"\n🚀 Starting upper-stream agent for query: {query}\n")
    
    final_state = None
    try:
        start_time = time.time()
        
        for event in app.stream(init_state, config=cfg):
            for v in event.values():
                final_state = v
                try:
                    content = v['messages'][-1].content
                    try:
                        obj = json.loads(content)
                        _vprint(json.dumps(obj, indent=2))
                    except json.JSONDecodeError:
                        _vprint(content)
                except Exception:
                    _vprint(str(v))
        
        end_time = time.time()
        _vprint(f"\n✅ Agent completed in {end_time - start_time:.2f} seconds")
        
    except GraphRecursionError as ge:
        _vprint(f"\n[GraphRecursionError] {ge}")
    except KeyboardInterrupt:
        _vprint("\n⚠️ Interrupted by user.")
    except Exception as e:
        _vprint(f"\n[ERROR] Agent execution failed: {e}")
        import traceback
        traceback.print_exc()
    
    # Extract final output
    if final_state and final_state.get('messages'):
        output = extract_final_output(final_state['messages'])
        output['session_directory'] = session_dir_abs
        output['original_query'] = query
        
        # Save summary
        summary_path = Path(session_dir_abs) / "agent_summary.json"
        try:
            with open(summary_path, "w") as f:
                json.dump(output, f, indent=2)
            _vprint(f"\n📄 Summary saved to: {summary_path}")
        except Exception as e:
            _vprint(f"\n[WARN] Could not save summary: {e}")
        
        return output
    
    return {"error": "No final state produced", "session_directory": session_dir_abs}

## Example: Run Agent

## Query Examples

### Date Range Filtering

You can filter literature by publication date using `min_date` and `max_date` parameters:

**Date Formats Supported:**
- Year only: `"2020"`
- Year and month: `"2020/06"`
- Full date: `"2020/06/15"`

**Examples:**

1. **Recent papers only (2020 onwards):**
   ```python
   tools = _build_tools(
       "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results",
       min_date="2020"
   )
   ```

2. **Specific date range:**
   ```python
   tools = _build_tools(
       "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results",
       min_date="2015/01",
       max_date="2020/12"
   )
   ```

3. **Papers before a certain date:**
   ```python
   tools = _build_tools(
       "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results",
       max_date="2019"
   )
   ```

**Note:** Date filters apply to PubMed and Europe PMC searches. Preprint servers (medRxiv, ChemRxiv) do not support date filtering through their APIs.

In [12]:
# Example: Search for recent transplant rejection papers (2020-2024)
# Rebuild tools with date range
agent_model_recent = _init_models()
tools_recent = _build_tools(
    "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results",
    min_date="2023",
    max_date="2025"
)

# Rebuild agent with date-filtered tools
memory_recent = MemorySaver()
agent_executor_recent = create_react_agent(
    agent_model_recent, 
    tools_recent, 
    prompt=UPPER_STREAM_AGENT_PROMPT, 
    checkpointer=memory_recent
)


/tmp/ipykernel_1769714/2488998679.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor_recent = create_react_agent(


In [13]:
# Example query options:

# 1. Hair compounds (original example)
# user_query = "form a database of compounds found in hairs"

# 2. Transplant rejection (with date filter - recent studies only)
# Note: Rebuild tools with date range for this query
# tools_recent = _build_tools(
#     "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results",
#     min_date="2020"  # Only papers from 2020 onwards
# )
# user_query = "find compounds that might occur in transplant rejection"

# 3. Environmental contaminants (specific date range)
# tools_2015_2020 = _build_tools(
#     "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results",
#     min_date="2015",
#     max_date="2020"
# )
# user_query = "identify environmental contaminants in blood samples"

# Default query (no date filter)
user_query = "form a database of compounds found in hairs"
try:
    result = run_upper_stream_agent(user_query)
finally:
    # ✅ Force-upload any pending spans/traces before the cell finishes
    wait_for_all_tracers()
    # If you also use the LangSmith Client somewhere, you can flush it too:
    # client.flush()
    print("✓ Traces flushed to LangSmith")
print("\n" + "="*60)
print("FINAL OUTPUT")
print("="*60)
print(json.dumps(result, indent=2))


📁 Session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/upper_stream_20251110_191809_387127_7523

🚀 Starting upper-stream agent for query: form a database of compounds found in hairs


----- Upper-Stream Agent Node Invoked -----


2025-11-10 11:18:10,606 - INFO - Using session directory: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/upper_stream_20251110_191809_387127_7523
2025-11-10 11:18:24,694 - INFO - Found 22 results total, 19 unique. Enriching...
2025-11-10 11:18:24,695 - INFO - Raw source counts before deduplication: {'EuropePMC:MED': 10, 'ChemRxiv': 2, 'PubMed': 10}
2025-11-10 11:18:24,694 - INFO - Found 22 results total, 19 unique. Enriching...
2025-11-10 11:18:24,695 - INFO - Raw source counts before deduplication: {'EuropePMC:MED': 10, 'ChemRxiv': 2, 'PubMed': 10}
2025-11-10 11:18:27,102 - INFO - [PMC OK] EuropePMC:MED: Ultrasensitive Direct Chemical Analysis of Human Hair Using 
2025-11-10 11:18:27,102 - INFO - [PMC OK] EuropePMC:MED: Ultrasensitive Direct Chemical Analysis of Human Hair Using 
2025-11-10 11:18:27,386 - INFO - [PMC OK] EuropePMC:MED: Skin Pressing: An Easy and Reliable Method to Induce Acute H
2025-11-10 11:18:27,386 - INFO - [PMC OK] EuropePMC:MED: Skin Pressing: 

[chemical_ner] Loading all JSON files from folder: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/upper_stream_20251110_191809_387127_7523
[chemical_ner] Found 19 JSON files in /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/upper_stream_20251110_191809_387127_7523
[chemical_ner] Loaded 19 text documents from folder
[chemical_ner] Processing 19 documents with chemical NER...
[chemical_ner] Using document batches of 25 to prevent memory issues
[chemical_ner] Processing document batch 1/1 (docs 1-19)


/home/chen/GNPS_SIM/agent/db_form_agent_system/.venv/lib/python3.10/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


2025-11-10 11:18:38,961 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine


/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/../code/chemical_ner_tool.py:240: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=use_fp16):


[model] ner -> cuda:0 (params on cuda:0), linker -> cuda:1 (params on None)
[NER] Preparing 19 documents for processing...
[NER]   Processed 19/19 docs, 65 chunks created
[NER] Preparation complete: 65 chunks in 0.15s
[NER] Running NER model on 65 chunks (batch_size=8)...


/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/../code/chemical_ner_tool.py:336: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=use_fp16):
Batch inference: 100%|██████████| 9/9 [00:00<00:00,  9.81it/s]
/home/chen/GNPS_SIM/agent/db_form_agent_system/notebooks/../code/chemical_ner_tool.py:346: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast(enabled=use_fp16):
Failed to find sentence: For peptide sequences, the top 100 sequences with the highest −10 lgP values from peptide analysis and sequences with a confidence score > 95% from novel peptide analysis were integrated and input into the Emboss database (https://www.ebi.ac.uk/Tools/seqstats/emboss-pepstats/, accessed on 23 May 2024) to obtain protein information Bas

[NER] NER complete in 0.98s
[NER] Running Entity Linker on 65 chunks (batch_size=16)...
[NER] Entity Linking complete in 153.18s
[timing-batch] chunks=65, ner=0.983s (15.1 ms/chunk), linker=153.178s (2356.6 ms/chunk), total=154.161s
[chemical_ner] NER+Linking completed in 173.77s (9.15s per doc)
[NER] Entity Linking complete in 153.18s
[timing-batch] chunks=65, ner=0.983s (15.1 ms/chunk), linker=153.178s (2356.6 ms/chunk), total=154.161s
[chemical_ner] NER+Linking completed in 173.77s (9.15s per doc)

----- Stopping: Agent produced NER deliverable -----
{
  "results_file": "/home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/upper_stream_20251110_191809_387127_7523/ner_results/chemical_ner_results.json",
  "mention_count": 313,
  "error": false
}

✅ Agent completed in 202.90 seconds

📄 Summary saved to: /home/chen/GNPS_SIM/agent/db_form_agent_system/searching_results/upper_stream_20251110_191809_387127_7523/agent_summary.json
✓ Traces flushed to LangSmith

FINAL OUTPUT
{
 